# 价格合适（Price is Right）—— 评估微调后的开源模型

## 练习目标

加载第 7 周用 **QLoRA** 训好的适配器（LoRA weights），在测试集上做推理，并用课程 `util.evaluate` 看平均绝对误差等指标。

## 怎么跑

1. Colab 配置 Secrets：`HF_TOKEN`
2. 确认 `HF_USER` / `RUN_NAME` 指向你自己 Hub 上的微调仓库
3. 从上到下运行；最后对比柱状图里各基线的 Error


In [ ]:
# ========== 安装依赖 + 拉取评估工具 util.py ==========
!pip install -q --upgrade bitsandbytes trl
# 课程 week7/util.py：内含 evaluate / Tester
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py


In [ ]:
# ========== 导入：推理与评估需要的库 ==========

# 标准库
import os
import re
import math
from tqdm import tqdm
# Colab Secrets
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
# 加载基座 + 分词器 + 量化配置；set_seed 保证可复现
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
# PeftModel：把 LoRA 适配器挂到量化基座上
from peft import PeftModel
# 课程提供的评估入口
from util import evaluate


In [ ]:
# ========== 常量：指向哪次微调 run ==========

# 与训练时相同的基座
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"
# Hub 用户名（须能读到下面的适配器仓库）
HF_USER = "denis-mutuma" # your HF name here!

LITE_MODE = True

DATA_USER = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

# lite：指定某次训练产生的 RUN_NAME；REVISION=None 表示用默认分支最新
if LITE_MODE:
  RUN_NAME = "2026-03-08_13.42.08-lite"
  REVISION = None
# 非 lite 时可改用下面示例（保持注释，不改当前控制流）
# else:
# RUN_NAME = "2025-11-28_18.47.07"
# REVISION = "b19c8bfea3b6ff62237fbb0a8da9779fc12cefbd"

# 拼出 Hub 上的适配器路径
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"


# ----- 推理侧量化开关 -----

QUANT_4_BIT = True
# 与训练相同：决定 compute dtype 用 bf16 还是 fp16
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8


### 登录 Hugging Face

若还没有账号：访问 https://huggingface.co 注册并创建 token。

在 Colab **Secrets**（左侧钥匙图标）添加名为 `HF_TOKEN` 的密钥。  
评估需要能下载基座模型以及你（或作者）推上去的 **PEFT 适配器**。


In [ ]:
# ========== 登录 Hugging Face ==========

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)


In [ ]:
# ========== 只取测试集（评估不需要重新训练）==========
dataset = load_dataset(DATASET_NAME)
test = dataset['test']


In [ ]:
# 看一条样本结构：通常含 prompt / completion 等字段
test[0]


## 现在加载分词器（Tokenizer）和模型（基座 + LoRA）


In [ ]:
# ========== BitsAndBytes 量化配置（与训练笔记本同结构）==========

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )


In [ ]:
# ========== 加载基座，再挂上 Hub 上的 PEFT 适配器 ==========

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# 若指定了 git revision，则按 commit 加载；否则用默认分支
if REVISION:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=REVISION)
else:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)


# 打印挂载适配器后的显存占用
print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")


In [ ]:
# 在笔记本里展示模型对象摘要（结构 / 设备等）
fine_tuned_model


# 关键时刻：推理模式下做价格预测

## 对照目标（课程里的大致水位）

- 「人类」水平约 **$87.62** 误差 —— 试着做得更好
- 或接近 **gpt-4.1-nano** 约 **$62.51**

## 注意

商品价格方差很大；模型**无法**凭空知道它没见过的信息（例如临时促销价）。解读指标时结合误差分布图，而不是只看单一数字。


In [ ]:
# ========== 单条样本预测函数：给 evaluate() 调用 ==========
def model_predict(item):
    # 把商品 prompt 编成张量并放到 GPU
    inputs = tokenizer(item["prompt"],return_tensors="pt").to("cuda")
    # 推理不需要梯度，省显存
    with torch.no_grad():
        # 只生成很短续写（价格数字通常很短）
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    # 去掉 prompt 前缀，只解码新生成的 token
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)


In [ ]:
# 固定随机种子，尽量复现生成行为
set_seed(42)
# 在测试集上跑评估：内部会算误差并画图
evaluate(model_predict, test)


In [ ]:
# ========== 用 Plotly 柱状图对比多种方法的 Error ==========
import plotly.graph_objects as go

# (方法名, 柱颜色, 误差数值) —— 数值来自作者实验记录，勿当「你机器上必现」
results = [
    ("NLP + LR", "gray", 75.25),
    ("Random Forest", "gray", 72.13),
    ("XGBoost", "gray", 65.32),
    ("Neural Network", "orange", 71.56),
    ("GPT 4.1 Nano", "slateblue", 86.54),
    ("GPT 5.1", "green", 41.01),
    ("GPT 4.1 Nano (Fine-tuned)", "red", 84.82),
    ("Llama-3.2-3B (QLoRA)", "blue", 61.79)
]

# 拆成三条序列，供 Bar 使用
labels, colors, values = zip(*results)

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))

fig.update_layout(
    title="Week 6 exercise - Denis Mutuma",
    yaxis=dict(range=[0, max(values)], title="Error"),
    xaxis=dict(tickangle=-45),
    width=1000,
    height=800
)

fig.show()
